# LongFlow — quality finish: 640 continuation + the polished render

Runtime: **A100**, ~2–2.5 h. Follows the quality-night results
(NOTES 2026-08-21): held-out WER was still FALLING at 40K (no overfit
signature) and the chunked renders were unpolished. Two cheap levers:

1. **Continue v3_640 from step 40K → 80K** (fresh Adam, cosine
   1.5e-4→1.5e-5, EMA re-seeded AT the trained weights — no dilution),
   checkpoints every 10K, held-out eval at each.
2. **Render the presumptive-best config for the first time:**
   v3_640(best step) + **audio-only teacher polish k=3, independent
   noise** + 30 s chunks — plus its unpolished twin for A/B.

Bundle → `quality_eval2.zip` (the scorer auto-picks the newest zip).


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Quality finish v1.0 (2026-08-21): 640 continuation + polished render"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, _CFGField
from src.flow_head.trainer import load_checkpoint, pairs_from_files, train

CACHE_V3_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v3"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/qfinish"
DRIVE_OUT = "/content/drive/MyDrive/longflow_quality"
EVAL_DIR = "/content/qfinish_eval"
for d in (OUT, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

LOCAL_CACHE = "/content/cache_v3"
n_drive = len(glob.glob(f"{CACHE_V3_DRIVE}/*.pt"))
if not os.path.exists(LOCAL_CACHE) or len(glob.glob(f"{LOCAL_CACHE}/*.pt")) < n_drive:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print(f"bulk-copying {n_drive} v3 files (~15 GB)...", flush=True)
    !cp {CACHE_V3_DRIVE}/*.pt {LOCAL_CACHE}/
print(f"{len(glob.glob(f'{LOCAL_CACHE}/*.pt'))} cache files local")

LOCAL_CKPT = "/content/ckpts"
os.makedirs(LOCAL_CKPT, exist_ok=True)
for name in ("v3_640_step40000.pt",):
    if not os.path.exists(f"{LOCAL_CKPT}/{name}"):
        shutil.copy(f"{CKPT_DIR}/{name}", f"{LOCAL_CKPT}/{name}")

if os.path.exists(f"{DRIVE_OUT}/qfinish_report.json"):
    with open(f"{DRIVE_OUT}/qfinish_report.json") as f:
        report = json.load(f)
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/qfinish_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt failed: {repr(e)[:120]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

def flow_cfg_render(head, mean_t, std_t, utt, seed=0):
    field = _CFGField(head, utt.neg_hidden.float().cuda(), 1.3)
    g = torch.Generator(device="cuda").manual_seed(seed)
    z = heun_sample(field, utt.hidden.float().cuda(), head.cfg.d_latent,
                    nfe=8, sway=0.0, generator=g)
    return decode_latents(z * std_t.cuda() + mean_t.cuda())

print("READY")


In [ ]:
# ===== Pool + identical held-out split, then CONTINUE 640: 40K -> 80K =====
HELD_OUT_PER_BIN = 5
all_files = sorted(glob.glob(f"{LOCAL_CACHE}/*.pt"))

def fname_bin(path):
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in all_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print(f"held-out {len(held_out_files)} / train {len(train_files)} (identical split to the ladder)")

data = pairs_from_files(train_files, dual_stream=False)  # same files -> identical mean/std
print(f"pool: {data.hidden.shape[0]} frames")

TAG = "v3_640b"
final = f"{CKPT_DIR}/{TAG}_step80000.pt"
if os.path.exists(final):
    print(f"{TAG}: final checkpoint already on Drive — skipping training")
else:
    # RAW weights (not EMA) for continuation; fresh EMA shadow re-seeds AT the
    # trained weights (EMA.__init__ clones current params — no zero-init dilution)
    head, _m, _s = load_checkpoint(f"{LOCAL_CKPT}/v3_640_step40000.pt", use_ema=False)
    print(f"continuing {TAG} from step 40000 ({head.param_count()/1e6:.2f}M params)")
    t0 = time.time()
    train(head, data, steps=40000, batch_size=1024, lr=1.5e-4, lr_final=1.5e-5,
          ema_decay=0.9999, device="cuda", log_every=2000, seed=1,
          checkpoint_every=10000,
          checkpoint_path_fn=lambda s: f"{CKPT_DIR}/{TAG}_step{s + 40000}.pt")
    print(f"done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


In [ ]:
# ===== Held-out renders per continuation checkpoint =====
SUBSET_PER_BIN = 2
FULL_EVAL_STEPS = {80000}
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}

for fpath in held_out_files:
    utt = load_utterance(fpath)
    tname = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{tname}"):
        sf.write(f"{EVAL_DIR}/{tname}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": tname, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
subset_files = [fs[:SUBSET_PER_BIN] for fs in held_out_by_bin.values()]
subset_files = [f for fs in subset_files for f in fs]

for step in (50000, 60000, 70000, 80000):
    key = f"{step}:v3_640b"
    p = f"{CKPT_DIR}/v3_640b_step{step}.pt"
    if key in manifest["checkpoints"] or not os.path.exists(p):
        continue
    head, mean_t, std_t = load_checkpoint(p)
    head = head.to("cuda")
    eval_files = held_out_files if step in FULL_EVAL_STEPS else subset_files
    entries = []
    for fpath in eval_files:
        utt = load_utterance(fpath)
        name = f"{utt.utt_id}_v3_640b_step{step}.wav"
        if not os.path.exists(f"{EVAL_DIR}/{name}"):
            sf.write(f"{EVAL_DIR}/{name}",
                     flow_cfg_render(head, mean_t, std_t, utt, seed=0), 24000)
        entries.append({"utt_id": utt.utt_id, "audio": name,
                        "teacher_audio": manifest["teacher"][utt.utt_id]["audio"],
                        "text": utt.text, "target_words": fname_bin(fpath),
                        "arm": "v3_640b"})
    manifest["checkpoints"][key] = entries
    del head
    torch.cuda.empty_cache()
    print(f"{key}: {len(entries)} renders", flush=True)

with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("held-out eval rendered")


## 4. The stack render — polished 640b, 30 s chunks (+ unpolished twin)

First-ever render of the presumptive-best config: continued 640 head,
heun8+CFG, **audio-only teacher polish k=3 with INDEPENDENT noise** (EMA
retired), 30 s chunks, 0.25 s crossfades. `qf_640b_plain` is the A/B twin.


In [ ]:
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        time.sleep(wait)
    raise RuntimeError(f"empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript_turns(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return turns

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
TURNS = turnscript_turns(ABL_WORDS)
CHUNKS, cur, cw = [], [], 0
for t in TURNS:
    cur.append(t); cw += len(t.split())
    if cw >= 80:
        CHUNKS.append("\n".join(cur) + "\n"); cur, cw = [], 0
if cur:
    CHUNKS.append("\n".join(cur) + "\n")
report["cl_script"] = "\n".join(TURNS) + "\n"
print(f"{len(CHUNKS)} chunks")

def polish(z, cond, neg, k, cfg_scale=1.3, total_steps=10):
    if k <= 0:
        return z
    sched = model.model.noise_scheduler
    sched.set_timesteps(total_steps)
    ts = sched.timesteps[-k:]
    zt = z.to("cuda", torch.bfloat16)
    cond2 = torch.cat([cond, neg], dim=0).to("cuda", torch.bfloat16)
    noise = torch.randn(zt.shape, device="cuda", dtype=torch.float32).to(torch.bfloat16)
    zt = sched.add_noise(zt, noise, ts[0].expand(zt.shape[0]))
    for t in ts:
        combined = torch.cat([zt, zt], dim=0)
        eps = model.model.prediction_head(
            combined, t.repeat(combined.shape[0]).to(combined), condition=cond2)
        c_eps, u_eps = torch.split(eps, len(eps) // 2, dim=0)
        guided = u_eps + cfg_scale * (c_eps - u_eps)
        zt = sched.step(guided, t, zt).prev_sample
    return zt.float()

class AudioPolishPatch(CFGFlowHeadPatch):
    def __init__(self, *args, audio_k=3, **kwargs):
        super().__init__(*args, **kwargs)
        self.audio_k = audio_k
        self.audio_latents = []

    def __enter__(self):
        super().__enter__()
        inner = self.model.sample_speech_tokens
        patch = self

        def flow_sample_ap(condition, neg_condition=None, cfg_scale=None):
            z = inner(condition, neg_condition=neg_condition, cfg_scale=cfg_scale)
            if neg_condition is not None and patch.audio_k > 0:
                za = polish(z.float(), condition.float().cuda(),
                            neg_condition.float().cuda(), patch.audio_k)
                patch.audio_latents.append(za.detach().cpu())
            else:
                patch.audio_latents.append(z.float().detach().cpu())
            return z  # loop untouched — audio path re-decoded after generation

        self.model.sample_speech_tokens = flow_sample_ap
        return self

def crossfade_stitch(wavs, sr=24000, fade_s=0.25):
    n = int(sr * fade_s)
    out = wavs[0]
    for wv in wavs[1:]:
        if len(out) < n or len(wv) < n:
            out = np.concatenate([out, wv])
            continue
        fade = np.linspace(0, 1, n, dtype=np.float32)
        out[-n:] = out[-n:] * (1 - fade) + wv[:n] * fade
        out = np.concatenate([out, wv[n:]])
    return out

BEST_STEP = 80000  # override after the scorer if an earlier ckpt wins
head_b, mean_b, std_b = load_checkpoint(f"{CKPT_DIR}/v3_640b_step{BEST_STEP}.pt")
head_b = head_b.to("cuda")

for tag, ak in (("qf_640b_polish3", 3), ("qf_640b_plain", 0)):
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    chunk_wavs = []
    for ci, chunk_text in enumerate(CHUNKS):
        torch.manual_seed(ci)
        with AudioPolishPatch(model, head_b, mean_b, std_b, nfe=8, sway=0.0,
                              sampler=heun_sample, audio_k=ak) as patch, \
             torch.inference_mode():
            gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                 tokenizer=processor.tokenizer,
                                 cfg_scale=1.3, max_new_tokens=600)
        if ak > 0:
            wv = decode_latents(torch.cat(patch.audio_latents))
        else:
            wv = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
        chunk_wavs.append(wv)
        print(f"  {tag} chunk {ci+1}/{len(CHUNKS)}", flush=True)
    save_wav(tag, crossfade_stitch(chunk_wavs), {"step": BEST_STEP, "audio_k": ak})
print("stack renders done — LISTEN: qf_640b_polish3 vs qf_640b_plain vs qc_teacher")


In [ ]:
# ===== Bundle -> Drive root (scorer auto-picks the newest quality zip) =====
import zipfile
with open(f"{EVAL_DIR}/quality_report.json", "w") as f:
    json.dump(report, f, indent=2)
ZIP = "/content/drive/MyDrive/quality_eval2.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_quality_gpu_colab.ipynb next")
